# logistic — Yaswanth
Read PRE_PROCESSING_DECISIONS.md and docs/TRAINING_GUIDE.md. Run 01 contains measured outputs. Review the results and add your own interpretation before submitting. Complete all five folds before comparing models.

In [1]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert (ROOT / "data").is_dir(), "Open this notebook from the repository root or notebooks directory"
os.chdir(ROOT)


In [2]:
get_ipython().run_line_magic("run", "notebooks/00_shared_training.ipynb")

In [3]:
MODEL = "logistic"
RUN_ID = "run_01"


## Optional smoke check
Set RUN_SMOKE=True to check the interface on 200 observations. These are not full experiment results.

In [4]:
RUN_SMOKE = False
if RUN_SMOKE:
    run(MODEL, "processed", f"scratch/{RUN_ID}", folds=[0], final=True, smoke=True)

## Full model comparison
This trains five trip-based CV fits and a final all-training fit, then evaluates student-test data. Preserve warnings and configuration. Use a fresh RUN_ID for another experiment.

In [5]:
run(MODEL, "processed", f"results/{RUN_ID}", folds=[0,1,2,3,4], final=True)

fold_0 {"split": "fold_0", "fit_seconds": 1.0751453470002161, "inference_seconds": 0.007320083001104649, "inference_ms_per_observation": 0.0002550551568329146, "metrics": {"all_points": {"observations": 28700, "top1_accuracy": 0.7775609756097561, "macro_precision": 0.6513185877985712, "macro_recall": 0.5957388917477748, "macro_f1": 0.6128575584949925, "candidate_miss_rate": 0.00975609756097561, "categorical_log_loss_with_miss_state": 1.0657711880274288}, "reliable_points": {"observations": 20087, "top1_accuracy": 0.858664808084831, "macro_precision": 0.7478084962913014, "macro_recall": 0.7023677863244904, "macro_f1": 0.7188867373866626, "candidate_miss_rate": 0.0010952357245979987, "categorical_log_loss_with_miss_state": 0.5150953997268584}, "reliable_coverage": 0.6998954703832753, "candidate_binary_log_loss": 0.17240481644189246, "route_continuity": null, "route_continuity_reason": "Needs raw road network and an agreed path/connectivity definition; do not invent a value."}, "fit_subse

## Inspect measured results

In [6]:
import json
output = Path("results") / RUN_ID / MODEL
print((output / "summary.json").read_text())
records = json.loads((output / "metrics.json").read_text())
for record in records:
    print(record["split"], record["metrics"]["all_points"], record["warnings"])

{
  "model": "logistic",
  "smoke": false,
  "cv_macro_f1_mean": 0.642805257133005,
  "cv_macro_f1_std": 0.048579439279387876,
  "completed_folds": 5,
  "student_test_completed": true
}
fold_0 {'observations': 28700, 'top1_accuracy': 0.7775609756097561, 'macro_precision': 0.6513185877985712, 'macro_recall': 0.5957388917477748, 'macro_f1': 0.6128575584949925, 'candidate_miss_rate': 0.00975609756097561, 'categorical_log_loss_with_miss_state': 1.0657711880274288} []
fold_1 {'observations': 40694, 'top1_accuracy': 0.7466211235071509, 'macro_precision': 0.7092323143701916, 'macro_recall': 0.6615752244391931, 'macro_f1': 0.6763544688436074, 'candidate_miss_rate': 0.10217722514375584, 'categorical_log_loss_with_miss_state': 3.3828167862742085} []
fold_2 {'observations': 49146, 'top1_accuracy': 0.4603223049688683, 'macro_precision': 0.6129129647471836, 'macro_recall': 0.5451518041366863, 'macro_f1': 0.566876339271367, 'candidate_miss_rate': 0.28549627640092784, 'categorical_log_loss_with_miss_

## Your interpretation
Write your own explanation of the algorithm/objective, hyperparameters, actual results, timing scope, fit and generalisation, convergence where applicable, and proxy-label limitations. Route continuity remains N/A until common graph scoring is implemented. Do not invent loss curves or results.

### Run 01 findings
Logistic regression learns a weighted sum of the ten candidate features and uses a sigmoid to estimate candidate correctness. It minimises regularised binary log loss. StandardScaler is fitted anew on the fitting trips of each fold. The model cannot directly express interactions unless they appear in the features.

Mean CV arc Macro F1 was 0.6428 (SD 0.0486). On 19,160 student-test points, Top-1 accuracy was 80.78%, arc Macro F1 was 0.6286 and categorical log loss was 0.7727. The five CV fits and final refit took 7.62 seconds of fitting. Student-test classifier inference averaged about 0.25 microseconds per observation in this batch.

### Compare the two baselines
Boosting's mean CV Macro F1 lead was about 0.0042, small relative to fold variability. On student test, boosting made one more correct Top-1 prediction, while logistic had higher arc Macro F1 and boosting had lower categorical log loss. The next three models are needed before choosing a tuning target. Compare meaningful fit and validation subsets before attributing any gap to overfitting. The labels are route-alignment proxies, and inference timing excludes preprocessing and HMM.
